<a href="https://colab.research.google.com/github/priyu9-star/BudgetWise-AI-based-Expense-Forecasting-Tool-Batch-6-Team-C-/blob/main/Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -q flask pandas numpy scikit-learn

from flask import Flask, jsonify, render_template_string
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import os
import warnings
warnings.filterwarnings("ignore")

# ---------------- CONFIG ----------------
CSV_NAME = "Personal_Finance_Dataset.csv"  # ensure this file is uploaded in Colab

# ---------------- HELPERS ----------------
def load_and_prepare(csv_path=CSV_NAME):
    # Validation: Check if file exists
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Upload your dataset named '{csv_path}' in the Colab Files pane before running this cell.")

    df = pd.read_csv(csv_path)

    # Validate required columns
    required_cols = ['Date', 'Type', 'Category', 'Amount']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    # Parse date (dataset uses DD-MM-YYYY)
    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')

    # Check for invalid dates
    if df['Date'].isna().any():
        print(f"Warning: {df['Date'].isna().sum()} rows have invalid dates and will be dropped")

    df = df.dropna(subset=['Date'])

    # Data validation
    if len(df) == 0:
        raise ValueError("No valid data remaining after cleaning")

    # Validate Amount column
    df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')
    if df['Amount'].isna().any():
        print(f"Warning: {df['Amount'].isna().sum()} rows have invalid amounts and will be set to 0")
    df['Amount'] = df['Amount'].fillna(0.0)

    # Check for negative amounts (could be refunds or corrections)
    negative_count = (df['Amount'] < 0).sum()
    if negative_count > 0:
        print(f"Note: {negative_count} transactions have negative amounts")

    df['Category'] = df['Category'].astype(str).str.strip()

    return df

def aggregate_past_months(exp_df):
    # Extract year and month first to avoid groupby issues
    exp_df = exp_df.copy()
    exp_df['year'] = exp_df['Date'].dt.year
    exp_df['month'] = exp_df['Date'].dt.month

    # Sum per year-month
    tmp = exp_df.groupby(['year', 'month'], as_index=False)['Amount'].sum()

    # Create label "Jan 2020"
    tmp['label'] = tmp.apply(
        lambda r: pd.Timestamp(year=int(r['year']),
                               month=int(r['month']),
                               day=1).strftime('%b %Y'),
        axis=1
    )

    # Sort
    tmp = tmp.sort_values(['year', 'month']).reset_index(drop=True)

    return tmp[['label', 'Amount']]

def predict_next_total(monthly_totals_df):
    # monthly_totals_df: dataframe with Amount
    # Build Month_Index from position to avoid gaps
    monthly_totals_df = monthly_totals_df.copy().reset_index(drop=True)

    # Check if we have enough data
    if len(monthly_totals_df) == 0:
        return 0.0
    if len(monthly_totals_df) == 1:
        return float(monthly_totals_df['Amount'].iloc[0])

    monthly_totals_df['Month_Index'] = np.arange(1, len(monthly_totals_df)+1)
    X = monthly_totals_df[['Month_Index']].values
    y = monthly_totals_df['Amount'].values

    try:
        model = LinearRegression().fit(X, y)
        next_idx = np.array([[monthly_totals_df['Month_Index'].max() + 1]])
        pred = float(model.predict(next_idx)[0])
        return round(max(pred, 0.0), 2)
    except Exception as e:
        print(f"Warning: Linear regression failed for total prediction: {e}")
        # Fallback: return average of last 3 months
        last_3_avg = monthly_totals_df['Amount'].tail(3).mean()
        return round(float(last_3_avg), 2)

def predict_next_by_category(df_exp):
    # Create continuous month index by Year+Month
    df = df_exp.copy()
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month

    # Check if we have data
    if len(df) == 0:
        return {}

    # Create proper month identifier
    df['ym'] = df['Year'].astype(str) + '-' + df['Month'].astype(str).str.zfill(2)

    # Ordered list of unique months
    months_sorted = sorted(df['ym'].unique())
    month_map = {m: i+1 for i, m in enumerate(months_sorted)}  # 1..N
    df['Month_Index'] = df['ym'].map(month_map)
    next_month_idx = max(month_map.values()) + 1 if month_map else 1

    grouped = df.groupby(['Category','Month_Index'], as_index=False)['Amount'].sum()
    categories = grouped['Category'].unique().tolist()
    preds = {}

    for cat in categories:
        cat_df = grouped[grouped['Category'] == cat].sort_values('Month_Index')
        X = cat_df[['Month_Index']].values
        y = cat_df['Amount'].values

        if len(X) == 0:
            preds[cat] = 0.0
            continue
        if len(X) == 1:
            preds[cat] = round(float(y[0]), 2)
            continue

        # Fit linear regression on cat time series with error handling
        try:
            model = LinearRegression().fit(X, y)
            pred = float(model.predict([[next_month_idx]])[0])
            preds[cat] = round(max(pred, 0.0), 2)
        except Exception as e:
            print(f"Warning: Linear regression failed for category '{cat}': {e}")
            # Fallback: use average of last available months
            last_avg = cat_df['Amount'].tail(3).mean()
            preds[cat] = round(float(last_avg), 2)

    return preds

# ---------------- FLASK APP ----------------
app = Flask(__name__)

@app.route('/api/forecast_all')
def api_forecast_all():
    try:
        df = load_and_prepare()
        # Filter only expenses
        expenses = df[df['Type'].str.lower() == 'expense'].copy()

        # Check if we have expense data
        if len(expenses) == 0:
            return jsonify({
                "error": "No expense data found in the dataset",
                "past_months": [],
                "past_totals": [],
                "total_next_month": 0.0,
                "category_next_month": {},
                "category_distribution_pct": {}
            })

        # Past monthly totals (labels like "Jan 2020")
        past_months_df = aggregate_past_months(expenses)
        past_labels = past_months_df['label'].tolist()
        past_totals = [round(float(x),2) for x in past_months_df['Amount'].tolist()]

        # Predict total next month using ALL past months
        total_next = predict_next_total(past_months_df)

        # Category-wise prediction using all historical months
        cat_preds = predict_next_by_category(expenses)

        # Also build category distribution percentages for pie chart
        total_pred = sum(cat_preds.values()) if cat_preds else 0.0
        cat_distribution = {k: (round((v/total_pred*100),2) if total_pred>0 else 0.0) for k,v in cat_preds.items()}

        # Return payload
        payload = {
            "past_months": past_labels,        # e.g. ["Jan 2020","Feb 2020",...]
            "past_totals": past_totals,        # corresponding totals
            "total_next_month": round(float(total_next),2),
            "category_next_month": cat_preds,
            "category_distribution_pct": cat_distribution
        }
        return jsonify(payload)

    except Exception as e:
        return jsonify({
            "error": str(e),
            "past_months": [],
            "past_totals": [],
            "total_next_month": 0.0,
            "category_next_month": {},
            "category_distribution_pct": {}
        })

# ---------------- FRONTEND HTML (Line + Bar + Pie) ----------------
html_page = """
<!doctype html>
<html>
<head>
  <meta charset="utf-8">
  <title>AI Expense Forecast Dashboard</title>
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <script src="https://cdn.tailwindcss.com"></script>
  <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
</head>
<body class="bg-gray-50 text-gray-900">
  <div class="max-w-6xl mx-auto p-6">
    <header class="mb-6">
      <h1 class="text-3xl font-bold text-indigo-600">AI Expense Forecast</h1>
      <p class="text-sm text-gray-600">Predicting next month's expenses using your historical data.</p>
      <div id="errorAlert" class="hidden mt-4 p-4 bg-red-100 border border-red-400 text-red-700 rounded"></div>
    </header>

    <div class="grid grid-cols-1 lg:grid-cols-3 gap-6 mb-6">
      <div class="col-span-2 bg-white p-4 rounded shadow">
        <h2 class="text-lg font-semibold mb-2">Total Expenses Over Time (past months + predicted)</h2>
        <canvas id="lineChart" height="120"></canvas>
      </div>

      <div class="bg-white p-4 rounded shadow">
        <h2 class="text-lg font-semibold">Next Month Prediction</h2>
        <p class="mt-3 text-3xl font-extrabold text-indigo-600" id="totalNext">Loading...</p>
        <p class="mt-2 text-sm text-gray-600">This is the total predicted expense for the next month.</p>
        <p class="mt-1 text-xs text-gray-500">Note: Uses linear regression; may not capture seasonal patterns.</p>
      </div>
    </div>

    <div class="grid grid-cols-1 md:grid-cols-2 gap-6 mb-6">
      <div class="bg-white p-4 rounded shadow">
        <h3 class="font-semibold mb-3">Category-wise Forecast (Bar)</h3>
        <canvas id="barChart" height="140"></canvas>
      </div>

      <div class="bg-white p-4 rounded shadow">
        <h3 class="font-semibold mb-3">Category Distribution (Pie)</h3>
        <canvas id="pieChart" height="140"></canvas>
      </div>
    </div>

    <div class="bg-white p-4 rounded shadow">
      <h3 class="font-semibold mb-3">Category Forecast Table</h3>
      <div class="overflow-x-auto">
        <table class="w-full text-left">
          <thead class="bg-gray-100 text-sm text-gray-600">
            <tr><th class="p-2">Category</th><th class="p-2">Predicted Amount (Next Month)</th></tr>
          </thead>
          <tbody id="catTable"></tbody>
        </table>
      </div>
    </div>

  </div>

<script>
async function load() {
  try {
    const res = await fetch('/api/forecast_all');
    const data = await res.json();

    // Check for errors from backend
    if (data.error) {
      document.getElementById('errorAlert').innerHTML = `<strong>Error:</strong> ${data.error}`;
      document.getElementById('errorAlert').classList.remove('hidden');
      return;
    }

    // Total next month
    document.getElementById('totalNext').innerText = "₹ " + Number(data.total_next_month).toLocaleString();

    // LINE CHART: past months + predicted next (append predicted label)
    const lineLabels = data.past_months.slice();
    const lineData = data.past_totals.slice();
    // predicted label: next month name attempt - show "Next Month"
    lineLabels.push("Next Month");
    lineData.push(data.total_next_month);

    const ctxLine = document.getElementById('lineChart').getContext('2d');
    new Chart(ctxLine, {
      type: 'line',
      data: {
        labels: lineLabels,
        datasets: [{
          label: 'Total Expense',
          data: lineData,
          borderColor: 'rgb(236,72,153)',
          backgroundColor: 'rgba(236,72,153,0.1)',
          tension: 0.25,
          pointRadius: 4,
          pointBackgroundColor: 'rgb(236,72,153)'
        }]
      },
      options: {
        responsive: true,
        scales: {
          y: {
            beginAtZero: true,
            ticks: { callback: v => '₹ ' + v }
          }
        }
      }
    });

    // BAR CHART: category predictions
    const categories = Object.keys(data.category_next_month);
    const catValues = Object.values(data.category_next_month);

    if (categories.length > 0) {
      const ctxBar = document.getElementById('barChart').getContext('2d');
      new Chart(ctxBar, {
        type: 'bar',
        data: {
          labels: categories,
          datasets: [{
            label: 'Predicted Amount',
            data: catValues,
            backgroundColor: 'rgba(99,102,241,0.8)'
          }]
        },
        options: {
          responsive: true,
          scales: {
            y: {
              beginAtZero: true,
              ticks: { callback: v => '₹ ' + v }
            }
          }
        }
      });

      // PIE CHART: distribution percent
      const pieLabels = categories;
      const pieValues = categories.map(c => data.category_next_month[c]);
      const total = pieValues.reduce((a,b) => a + b, 0) || 1;

      const ctxPie = document.getElementById('pieChart').getContext('2d');
      new Chart(ctxPie, {
        type: 'pie',
        data: {
          labels: pieLabels,
          datasets: [{
            data: pieValues,
            backgroundColor: pieLabels.map((_, i) => `hsl(${i * 40 % 360} 70% 60%)`)
          }]
        },
        options: {
          responsive: true,
          plugins: {
            tooltip: {
              callbacks: {
                label: (ctx) => `${ctx.label}: ₹ ${ctx.parsed} (${(ctx.parsed/total*100).toFixed(1)}%)`
              }
            }
          }
        }
      });

      // Populate table
      const tbody = document.getElementById('catTable');
      tbody.innerHTML = '';
      categories.forEach(cat => {
        const tr = document.createElement('tr');
        tr.className = 'border-t';
        tr.innerHTML = `<td class="p-2">${cat}</td><td class="p-2 font-semibold">₹ ${Number(data.category_next_month[cat]).toLocaleString()}</td>`;
        tbody.appendChild(tr);
      });
    } else {
      document.getElementById('barChart').innerHTML = '<p class="text-gray-500 text-center py-8">No category data available</p>';
      document.getElementById('pieChart').innerHTML = '<p class="text-gray-500 text-center py-8">No category data available</p>';
      document.getElementById('catTable').innerHTML = '<tr><td colspan="2" class="p-2 text-center text-gray-500">No category data available</td></tr>';
    }
  } catch (error) {
    console.error('Error loading data:', error);
    document.getElementById('errorAlert').innerHTML = `<strong>Error:</strong> Failed to load data: ${error.message}`;
    document.getElementById('errorAlert').classList.remove('hidden');
  }
}

load();
</script>
</body>
</html>
"""

@app.route('/')
def home():
    return render_template_string(html_page)

# ---------------- RUN IN COLAB ----------------
print("Starting Flask server in Colab...")
print("\nTo access your dashboard:")
print("1. Look for the output that says 'Running on http://127.0.0.1:5000'")
print("2. Click on the web preview button (globe icon) in the Colab toolbar")
print("3. Select 'Preview on port 5000'")
print("\nThe dashboard will load in a new tab.")

# Run the app
if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)

Starting Flask server in Colab...

To access your dashboard:
1. Look for the output that says 'Running on http://127.0.0.1:5000'
2. Click on the web preview button (globe icon) in the Colab toolbar
3. Select 'Preview on port 5000'

The dashboard will load in a new tab.
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [27/Nov/2025 13:25:51] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Nov/2025 13:25:52] "GET /api/forecast_all HTTP/1.1" 200 -


In [ ]:
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(5000)


Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

df = pd.read_csv("Personal_Finance_Dataset.csv")
print(df.columns.tolist())
df.head()


['Date', 'Transaction Description', 'Category', 'Amount', 'Type']


,Date,Transaction Description,Category,Amount,Type
0,02-01-2020,Score each.,Food & Drink,1485.69,Expense
1,02-01-2020,Quality throughout.,Utilities,1475.58,Expense
2,04-01-2020,Instead ahead despite measure ago.,Rent,1185.08,Expense
3,05-01-2020,Information last everything thank serve.,Investment,2291.00,Income
4,13-01-2020,Future choice whatever from.,Food & Drink,1126.88,Expense
